New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys, inspect
print('kernel_cwd:', Path.cwd())
print('project_present:', Path('/home/nahisaho/GitHub/jupytermind/src/ai_data_scientist').is_dir())
import ai_data_scientist
print('module:', ai_data_scientist.__file__)
from ai_data_scientist import project_manager as pm, lifecycle, language_router
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
handle = pm.resolve_project('kaggle-v020-kaggle-exp-01-titanic')
pm.ensure_notebook(handle)
print('target:', handle.notebook_path)
print('language:', language_router.detect_language('Titanicの生存者と非生存者の違いを日本語で分析してください。'))
print('bootstrap_ready')

kernel_cwd: /home/nahisaho/GitHub/jupytermind
project_present: True
module: /home/nahisaho/GitHub/jupytermind/src/ai_data_scientist/__init__.py
target: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-01-titanic/notebooks/kaggle-v020-kaggle-exp-01-titanic.ipynb
language: ja
bootstrap_ready


In [2]:
import nbformat
from ai_data_scientist import lifecycle
RUN_ID = 'v020-exp-01'
lifecycle.register_run(RUN_ID, handle.notebook_path)
plan = '''# Titanic：生存者と非生存者の比較（v020-exp-01）

## 分析計画
対象はKaggle Titanic競技のラベル付きtrain相当の公開データ。Kaggle SDKで検索し、ファイル一覧と構造・人数・ID範囲を確認する。testやgender_submissionの予測ラベルは混ぜない。外部CSVミラーへはフォールバックしない。

1. 出所（owner/dataset slug、ファイル、取得日時、SHA-256）と列定義・前提を記録。欠損、重複、値域・意味的異常を検査する。
2. 生存/非生存の人数、性別、客室等級、年齢、運賃、同乗家族数、乗船港、客室情報の有無を比較。カテゴリには分母付き生存率とWilson 95%区間、連続値には中央値/IQRと分布を用いる。
3. 性別×等級の層別比較、調整ロジスティック回帰、欠損/外れ値/年齢区分の感度分析を行う。p値の羅列ではなく差の大きさと不確実性を重視する。
4. 初回結果を読み直し、重要な未解決点があれば自然言語の追加依頼を原文保存し最大3回実行。選定理由・結果・証拠・残る限界を記録する。
5. カーネル再起動後、全コードセルを上からJupyter MCPで再実行しvisual_audit=Trueとライフサイクル状態を保存する。

## 解釈と実行経路
これは観測された学習用部分標本の記述・関連分析であり因果推論ではない。乗客・家族・同一券で観測が依存する可能性、学習用標本の選択、運賃の測定単位、年齢欠損に注意する。分析とKaggle API呼出しはJupyter MCPのみ。mcp_gateway.run_and_recordはカーネル内から同じカーネルを再入実行するクライアントを提供しないため使用せず、実際のJupyter MCP execute_cellを使う（擬似クライアントやexecによる代用なし）。Notebookの構成・証拠・メタデータの明示的変更にはproject_manager.enqueue_writeを使い、セル実行結果はJupyter MCPが保存する。トークン・認証設定は表示しない。
'''
setup = '''from pathlib import Path
import os, io, json, hashlib, zipfile, re, warnings
from datetime import datetime, timezone
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from dataclasses import asdict
import numpy as np
import pandas as pd
import nbformat
from IPython.display import display
from ai_data_scientist import project_manager as pm, lifecycle, language_router
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
handle = pm.resolve_project('kaggle-v020-kaggle-exp-01-titanic')
pm.ensure_notebook(handle)
data_dir = pm.ensure_data_dir(handle)
RUN_ID = 'v020-exp-01'
lifecycle.register_run(RUN_ID, handle.notebook_path)
phase_log = []
@contextmanager
def phase(label, writing=False):
    if lifecycle.is_cancel_requested(RUN_ID):
        lifecycle.mark_failed(RUN_ID)
        raise RuntimeError('キャンセル要求により停止')
    start = lifecycle.mark_write_start if writing else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if writing else lifecycle.mark_execution_end
    start(RUN_ID)
    phase_log.append({'phase': label, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        raise
    finally:
        end(RUN_ID)
        phase_log.append({'phase': label, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
def safe_api(fn):
    failure = None
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        try:
            result = fn()
        except Exception as exc:
            failure = type(exc).__name__
    if failure:
        raise RuntimeError('Kaggle API失敗（詳細は認証情報保護のため非表示）: ' + failure) from None
    return result
print('run_id:', RUN_ID, 'language:', language_router.detect_language('生存者と非生存者の違いを日本語で調べる'))
print('Notebook:', handle.notebook_path.relative_to(ROOT))
print(asdict(lifecycle.get_run_status(RUN_ID)))
'''
acquisition = '''with phase('Kaggle公開データ検索・ファイル一覧確認'):
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    safe_api(lambda: api.authenticate())
    candidates = safe_api(lambda: api.dataset_list(search='titanic', page=1))
    refs = [str(d.ref) for d in candidates]
    print('Kaggle検索結果:', refs)
    chosen_slug = None
    chosen_files = None
    preferred = ['heptapod/titanic', 'yasserh/titanic-dataset']
    ordered = [s for s in preferred if s in refs] + [s for s in refs if s not in preferred]
    for slug in ordered[:12]:
        listing = safe_api(lambda slug=slug: api.dataset_list_files(slug))
        file_names = [f.name for f in listing.files]
        print('ファイル一覧:', slug, file_names)
        train_files = [f for f in file_names if f.rsplit('/', 1)[-1].lower() == 'train.csv']
        if train_files:
            chosen_slug, chosen_files = slug, train_files
            break
    if chosen_slug is None:
        raise RuntimeError('対応する公開train.csvをKaggle APIで確認できず停止。外部ソースへのフォールバックなし。')
    file_name = chosen_files[0]
    raw_dir = data_dir / 'kaggle_raw'
    raw_dir.mkdir(parents=True, exist_ok=True)
    safe_api(lambda: api.dataset_download_file(chosen_slug, file_name, path=str(raw_dir), force=True, quiet=True))
    plain = raw_dir / Path(file_name).name
    zipped = raw_dir / (Path(file_name).name + '.zip')
    if plain.is_file() and zipfile.is_zipfile(plain):
        archive = plain
    elif zipped.is_file():
        archive = zipped
    else:
        archive = None
    if archive is not None:
        with zipfile.ZipFile(archive) as z:
            matches = [n for n in z.namelist() if Path(n).name == Path(file_name).name]
            if len(matches) != 1:
                raise RuntimeError('ZIP内の対応CSVを一意に選べず停止')
            payload = z.read(matches[0])
    elif plain.is_file():
        payload = plain.read_bytes()
    else:
        raise RuntimeError('Kaggle SDKの取得ファイルが確認できず停止')
    csv_path = data_dir / 'train.csv'
    csv_path.write_bytes(payload)
    from ai_data_scientist.ingestion import ingest
    df = ingest({'type': 'csv', 'path': str(csv_path)})
    print('取得データ型:', type(df).__name__)
    provenance = {'competition_reference': 'https://www.kaggle.com/competitions/titanic', 'owner_dataset_slug': chosen_slug, 'retrieved_file': file_name, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(payload).hexdigest(), 'bytes': len(payload)}
    print(json.dumps(provenance, ensure_ascii=False, indent=2))
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    display(df.head(3).drop(columns=['Name', 'Ticket', 'Cabin'], errors='ignore'))
'''
def init_target(nb):
    if nb.cells:
        raise RuntimeError('既存Notebookは上書きせず停止')
    nb.cells = [nbformat.v4.new_markdown_cell(plan), nbformat.v4.new_code_cell(setup, id='setup'), nbformat.v4.new_code_cell(acquisition, id='acquisition')]
    nb.metadata['run_id'] = RUN_ID
lifecycle.mark_write_start(RUN_ID)
try:
    pm.enqueue_write(handle, init_target)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('分析計画・初期コードをenqueue_writeで保存')

分析計画・初期コードをenqueue_writeで保存


In [3]:
def fix_ingestion(nb):
    cell = next(c for c in nb.cells if c.id == 'acquisition')
    cell.source = cell.source.replace('from ai_data_scientist.ingestion import ingest', 'from ai_data_scientist.ingestion import ingest, SourceSpec').replace("df = ingest({'type': 'csv', 'path': str(csv_path)})", "ingestion_result = ingest(SourceSpec(kind='csv', location=str(csv_path)))\n    df = ingestion_result.dataframe\n    assert not ingestion_result.truncated")
lifecycle.mark_write_start(RUN_ID)
try:
    pm.enqueue_write(handle, fix_ingestion)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('ingestionの実シグネチャに合わせて保存')

ingestionの実シグネチャに合わせて保存
